# QCheck

In [2]:
#use "topfind";;
#require "qcheck";;
#use "../L09-qcheck/codice/esercizi.ml";;


- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads


- : unit = ()


/home/m1/.opam/default/lib/ocaml/unix: added to search path
/home/m1/.opam/default/lib/ocaml/unix/unix.cma: loaded
/home/m1/.opam/default/lib/qcheck-core: added to search path
/home/m1/.opam/default/lib/qcheck-core/qcheck_core.cma: loaded
/home/m1/.opam/default/lib/qcheck-core/runner: added to search path
/home/m1/.opam/default/lib/qcheck-core/runner/qcheck_runner.cma: loaded
/home/m1/.opam/default/lib/seq: added to search path
/home/m1/.opam/default/lib/stdlib-shims: added to search path
/home/m1/.opam/default/lib/stdlib-shims/stdlib_shims.cma: loaded
/home/m1/.opam/default/lib/ounit2/advanced: added to search path
/home/m1/.opam/default/lib/ounit2/advanced/oUnitAdvanced.cma: loaded
/home/m1/.opam/default/lib/ounit2: added to search path
/home/m1/.opam/default/lib/ounit2/oUnit.cma: loaded
/home/m1/.opam/default/lib/qcheck-ounit: added to search path
/home/m1/.opam/default/lib/qcheck-ounit/qcheck_ounit.cma: loaded
/home/m1/.opam/default/lib/qcheck: added to search path
/home/m1/.opam/d


val a_list : int list QCheck2.Gen.t = <abstr>

val pl : int list QCheck2.Print.t = <fun>

val compress : int list -> (int * int) list = <fun>

val gen_sorted : int list QCheck2.Gen.t = <abstr>

val example_3 : QCheck2.Test.t = Test.Test <abstr>


## Esercizio 1 - `compress` e le proprietà

Implementa `compress`, che trasforma gli elementi uguali consecutivi nella coppia `(elemento, numero di occorrenze)`: per esempio, `[1; 1; 2; 1]` diventa `[(1, 2); (2, 1); (1, 1)]`.

Prima di guardare un'implementazione, scrivi tre proprietà: una della famiglia «andata e ritorno», un'invariante e una che metta in relazione `compress` con una funzione che conosci già. Poi verifica le proprietà con QCheck.

1) andata e ritorno: it doesnt involute but -->
decompress (compress a' list) = a' list

esclude alterazioni agli elementi

2) invariante: lengt compress a' list = lengt compress list_rev a' list

esclude aggiungere di elementi ?

3) oracle: not possible

4) metamorfica:  map (x: int --> -x: int) compress(-a' list) = compress (a' list)


5) gestione input: EMPTY list
compress []=[]

In [3]:
let compress (xs : int list) : (int * int) list =
  let rec loop acc = function
    | [] -> List.rev acc
    | x :: rest ->
        let rec count n = function
          | y :: tail when x = y -> count (n + 1) tail
          | tail -> n, tail
        in
        let n, tail = count 1 rest in
        loop ((x, n) :: acc) tail
  in
  loop [] xs

let decompress (runs : (int * int) list) : int list =
  List.concat_map (fun (x, n) -> List.init n (fun _ -> x)) runs

let list_rev (xs : int list) : int list =
  List.rev xs

let negate_runs (runs : (int * int) list) : (int * int) list =
  List.map (fun (x, n) -> (-x, n)) runs

let metamorfica (xs : int list) : bool =
  negate_runs (compress (List.map (fun x -> -x) xs)) = compress xs

val compress : int list -> (int * int) list = <fun>


val decompress : (int * int) list -> int list = <fun>


val list_rev : int list -> int list = <fun>


val negate_runs : (int * int) list -> (int * int) list = <fun>


val metamorfica : int list -> bool = <fun>


In [7]:
(* Generatore: liste di interi, comprese liste vuote e valori negativi. *)
let compress_input = Gen.list Gen.int;;

(* Esempi fissi e casi limite. *)
let compress_examples =
  Test.make ~name:"compress: esempi e casi limite"
    ~print:pl (Gen.return []) (fun _ ->
      compress [] = []
      && compress [ 7 ] = [ (7, 1) ]
      && compress [ 1; 1; 2; 1 ] = [ (1, 2); (2, 1); (1, 1) ]);;

(* Proprietà 1: andata e ritorno. *)
let compress_roundtrip =
  Test.make ~count:10_000 ~name:"compress: andata e ritorno"
    ~print:pl compress_input (fun xs ->
      decompress (compress xs) = xs);;

(* Proprietà 2: invariante sulla lunghezza dei blocchi. *)
let compress_invariant =
  Test.make ~count:10_000 ~name:"compress: invariante lunghezza"
    ~print:pl compress_input (fun xs ->
      List.length (compress (List.rev xs)) = List.length (compress xs));;

(* Proprietà 3: proprietà metamorfica rispetto alla negazione. *)
let compress_metamorphic =
  Test.make ~count:10_000 ~name:"compress: proprietà metamorfica"
    ~print:pl compress_input (fun xs ->
      negate_runs (compress xs)
      = compress (List.map (fun x -> -x) xs));;

QCheck_base_runner.run_tests
  [ compress_examples; compress_roundtrip; compress_invariant;
    compress_metamorphic ];;

val compress_input : int list QCheck2.Gen.t = <abstr>


val compress_examples : QCheck2.Test.t = Test.Test <abstr>


val compress_roundtrip : QCheck2.Test.t = Test.Test <abstr>


val compress_invariant : QCheck2.Test.t = Test.Test <abstr>


val compress_metamorphic : QCheck2.Test.t = Test.Test <abstr>


success (ran 4 tests)


- : int = 0


## Esercizio 2 - trovare proprietà insufficienti

Scrivi una versione volutamente sbagliata di `compress` che superi le tre proprietà dell'esercizio precedente. Se non ci riesci, spiega perché le tue proprietà sono sufficienti. Se ci riesci, aggiungi la proprietà che rivela l'errore.

la scorsa versione era sbagliata, mancava la proprietà di essere effettivamente una compress:

6) compress compress [x; x] = [(x, 2)]

but it doesnt scale well, it's only a degeneralization of:

6.1) ∀ x : int, n ≥ 1:  compress (List.init n (fun _ -> x)) = [(x, n)]

In [10]:
(* Versione volutamente sbagliata: conserva ogni elemento come blocco singolo,
   quindi ricostruisce la lista ma non unisce le ripetizioni consecutive. *)
let wrong_compress (xs : int list) : (int * int) list =
  List.map (fun x -> (x, 1)) xs;;

val wrong_compress : int list -> (int * int) list = <fun>


In [11]:
(* Riapplichiamo le tre proprietà dell'esercizio 1 a wrong_compress. *)
let wrong_roundtrip =
  Test.make ~count:10_000 ~name:"wrong_compress: andata e ritorno"
    ~print:pl compress_input (fun xs ->
      decompress (wrong_compress xs) = xs);;

let wrong_invariant =
  Test.make ~count:10_000 ~name:"wrong_compress: invariante lunghezza"
    ~print:pl compress_input (fun xs ->
      List.length (wrong_compress (List.rev xs))
      = List.length (wrong_compress xs));;

let wrong_metamorphic =
  Test.make ~count:10_000 ~name:"wrong_compress: proprietà metamorfica"
    ~print:pl compress_input (fun xs ->
      negate_runs (wrong_compress xs)
      = wrong_compress (List.map (fun x -> -x) xs));;

(* Nuovo vincolo: una ripetizione consecutiva deve diventare un blocco
   con conteggio almeno 2. Il generatore garantisce la ripetizione. *)
let wrong_shape =
  Test.make ~count:10_000 ~name:"wrong_compress: unisce i duplicati"
    ~print:pl repeated_input (fun xs ->
      match wrong_compress xs with
      | (_, count) :: _ -> count >= 2
      | [] -> false);;

(* I primi tre passano; il quarto deve fallire e mostrare il difetto. *)
QCheck_base_runner.run_tests
  [ wrong_roundtrip; wrong_invariant; wrong_metamorphic; wrong_shape ];;

val wrong_roundtrip : QCheck2.Test.t = Test.Test <abstr>


val wrong_invariant : QCheck2.Test.t = Test.Test <abstr>


val wrong_metamorphic : QCheck2.Test.t = Test.Test <abstr>


val wrong_shape : QCheck2.Test.t = Test.Test <abstr>



--- Failure --------------------------------------------------------------------

Test wrong_compress: unisce i duplicati failed (4 shrink steps):

[0; 0]
failure (1 tests failed, 0 tests errored, ran 4 tests)


- : int = 1


## Esercizio 3 - generare liste ordinate

Per verificare una funzione che si aspetta una lista già ordinata, non usare `Gen.list`: quasi nessuna lista casuale è ordinata. Scrivi un generatore di liste ordinate partendo da una lista qualsiasi e ordinandola con `Gen.map`. Poi indica quali liste ordinate non possono essere generate in questo modo.

In [3]:
let gen_sorted : int list Gen.t =
  Gen.map (List.sort compare) (Gen.list Gen.int)



val gen_sorted : int list QCheck2.Gen.t = <abstr>


In [4]:
let sorted_test =
  Test.make ~count:1000 ~name:"il generatore produce liste ordinate"
    ~print:pl gen_sorted
    (fun l -> l = List.sort compare l);;

QCheck_base_runner.run_tests [ sorted_test ];;


val sorted_test : QCheck2.Test.t = Test.Test <abstr>


random seed: 374618437
success (ran 1 tests)



- : int = 0


## Esercizio 4 - progettare le proprietà

Senza scrivere codice, spiega in tre righe cosa avresti dovuto chiederti prima di scrivere le proprietà per `fake_sort`, così da accorgerti che due test su tre passavano anche per una funzione sbagliata. La risposta non è «provare più casi».

1) Per ogni proprietà mi sarei chiesto: «quale funzione sbagliata la soddisfa lo stesso?»
   (es.: `fun _ -> []` è ordinata; l'identità conserva lunghezza ed elementi).
2) I test che passavano per la funzione sbagliata controllavano una sola delle due promesse
   di "ordinare" (la forma dell'output, o la lunghezza e gli elementi) e nessuno controllava
   l'altra: ordinare promette che l'output sia ordinato E che contenga gli stessi elementi.
3) Prima di fermarmi mi sarei chiesto: «le proprietà insieme lasciano un solo output possibile
   per ogni input?». Se ne trovo due, manca una proprietà.